# SegFormer fine-tune on Colab — 차선 세그 (Roboflow 서버 직접 다운로드, Drive 불필요)

Phase 1: BEV 차선 세그용 SegFormer fine-tune.
**3 클래스 + 배경** → id2label = {0: background, 1: left-solid, 2: right-solid, 3: center-dashed}

이 id2label 순서는 `data_pipeline/extract_labels.py`의 `SegFormerLaneSeg`가
기대하는 것과 정확히 일치해야 한다 (배경=0, 의미 클래스 1..3 → 출력 채널 0..2).

**Drive 마운트 없음. 이미지 업로드 없음.** Roboflow API로 세그 데이터셋을 바로 내려받는다.

사용 순서:
1. **런타임 → GPU (T4/L4)**
2. Roboflow에서 BEV jpg 업로드 + polygon 세그 라벨링(좌실선/우실선/중앙점선) 후,
   **Generate → Export → 포맷 `PNG Mask Semantic` (또는 COCO Segmentation)** 으로 version 생성
3. `CONFIG` 채우고 실행
4. 끝나면 체크포인트 폴더(`segformer_lane/`)를 zip으로 받아 압축해제 후
   `extract_labels.py --segformer_ckpt segformer_lane` 로 사용

> 이 노트북은 Roboflow의 **Semantic Segmentation Mask (PNG)** export를 가정한다.
> export하면 `train/`, `valid/` 아래에 원본 이미지와 `*_mask.png`(클래스 인덱스맵)가 생긴다.

## 1. 설치

In [ ]:
!pip install -q transformers datasets roboflow evaluate
import torch, transformers
print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. CONFIG

`ID2LABEL` 순서 고정 — extract_labels.py contract. 클래스 색/순서를 Roboflow에서
동일하게 맞춰야 한다 (배경 0, 좌실선 1, 우실선 2, 중앙점선 3).

In [ ]:
RF_API_KEY   = 'YOUR_API_KEY'
RF_WORKSPACE = 'your-workspace'
RF_PROJECT   = 'your-lane-project'
RF_VERSION   = 1
RF_FORMAT    = 'png-mask-semantic'   # Roboflow semantic mask export

BASE_MODEL = 'nvidia/segformer-b0-finetuned-ade-512-512'  # b0 = Jetson 가벼움
IMG_SIZE   = 512        # SegFormer 학습 입력 (추론은 224 BEV에 upsample되므로 무관)
EPOCHS     = 80
BATCH      = 8
LR         = 6e-5
OUT_DIR    = '/content/segformer_lane'

# extract_labels.SegFormerLaneSeg와 반드시 일치
ID2LABEL = {0: 'background', 1: 'left-solid', 2: 'right-solid', 3: 'center-dashed'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)
print('id2label:', ID2LABEL)

## 3. Roboflow 데이터셋 다운로드

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download(RF_FORMAT)
print('location:', dataset.location)

import os, glob
for split in ('train', 'valid'):
    d = os.path.join(dataset.location, split)
    if os.path.isdir(d):
        print(split, ':', len(glob.glob(os.path.join(d, '*'))), 'files')

## 4. Dataset — 이미지 + 마스크 PNG 로드

Roboflow PNG-mask-semantic export는 이미지와 클래스 인덱스 마스크를 같이 준다.
마스크 파일명 규칙(`<img>_mask.png` 등)은 export마다 다를 수 있어, 아래 셀에서
실제 파일 한 쌍을 출력해 확인한 뒤 `mask_path_for()`를 맞춰라.

In [ ]:
import glob
train_dir = os.path.join(dataset.location, 'train')
files = sorted(glob.glob(os.path.join(train_dir, '*')))
print('sample files:')
for f in files[:6]:
    print(' ', os.path.basename(f))
# → 이미지와 마스크 네이밍 규칙 확인 후 아래 mask_path_for 수정

In [ ]:
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset

def list_pairs(split_dir):
    """(image_path, mask_path) 쌍 목록. Roboflow png-mask-semantic 규칙:
    원본은 .jpg, 마스크는 동일 stem + '_mask.png'. export별로 다르면 여기 수정."""
    pairs = []
    for img in sorted(glob.glob(os.path.join(split_dir, '*.jpg'))):
        stem = os.path.splitext(img)[0]
        mask = stem + '_mask.png'
        if os.path.exists(mask):
            pairs.append((img, mask))
    return pairs

class LaneSegDataset(Dataset):
    def __init__(self, pairs, processor):
        self.pairs = pairs
        self.processor = processor
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_path, mask_path = self.pairs[i]
        image = Image.open(img_path).convert('RGB')
        mask  = Image.open(mask_path)  # 클래스 인덱스맵 (0..3)
        enc = self.processor(image, mask, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

from transformers import SegformerImageProcessor
processor = SegformerImageProcessor.from_pretrained(BASE_MODEL, do_reduce_labels=False)
processor.size = {'height': IMG_SIZE, 'width': IMG_SIZE}

train_pairs = list_pairs(os.path.join(dataset.location, 'train'))
val_pairs   = list_pairs(os.path.join(dataset.location, 'valid'))
print('train pairs:', len(train_pairs), '| val pairs:', len(val_pairs))
assert len(train_pairs) > 0, 'no (image, mask) pairs found — list_pairs 규칙 확인'

train_ds = LaneSegDataset(train_pairs, processor)
val_ds   = LaneSegDataset(val_pairs, processor)

## 5. 모델 + Trainer

In [ ]:
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer

model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,   # ADE20K head → 4클래스 head 교체
)

args = TrainingArguments(
    output_dir='/content/seg_ckpts',
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

In [ ]:
import evaluate
metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    # logits: (B, C, h, w) → 라벨 해상도로 upsample 후 argmax
    upsampled = torch.nn.functional.interpolate(
        logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
    preds = upsampled.argmax(dim=1).numpy()
    res = metric.compute(predictions=preds, references=labels,
                         num_labels=NUM_LABELS, ignore_index=255,
                         reduce_labels=False)
    return {'mean_iou': res['mean_iou'], 'mean_acc': res['mean_accuracy']}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

## 6. 체크포인트 저장 (extract_labels.py가 from_pretrained로 로드)

`model.save_pretrained` + `processor.save_pretrained`를 같은 폴더에 저장해야
`SegFormerLaneSeg(checkpoint_path)`가 그대로 로드한다.

In [ ]:
import shutil
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print('saved checkpoint ->', OUT_DIR)
print(os.listdir(OUT_DIR))   # config.json, model.safetensors, preprocessor_config.json 등

# zip으로 묶어 다운로드 (Drive 안 씀)
shutil.make_archive('/content/segformer_lane', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/segformer_lane.zip')

## 7. (선택) sanity check — 한 장 추론 시각화

In [ ]:
import matplotlib.pyplot as plt
model.eval()
img_path, _ = val_pairs[0]
image = Image.open(img_path).convert('RGB')
inp = processor(image, return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**inp).logits
up = torch.nn.functional.interpolate(logits, size=image.size[::-1],
                                     mode='bilinear', align_corners=False)
pred = up.argmax(1)[0].cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image); ax[0].set_title('BEV'); ax[0].axis('off')
ax[1].imshow(pred, vmin=0, vmax=NUM_LABELS-1, cmap='tab10')
ax[1].set_title('pred (0 bg / 1 L / 2 R / 3 dash)'); ax[1].axis('off')
plt.show()